# 04 - Segmentation client (Phase 5)

**Objectif :** identifier des groupes de clients homogènes et interprétables à partir des variables RFM et comportementales, sans fuite de données.

> Prérequis : `python run_pipeline.py` pour produire `data/processed/customer_features.csv`.  
> Les modèles sont sauvegardés dans `models/` (créé automatiquement).  
> UMAP est optionnel (`pip install umap-learn`) ; t-SNE est utilisé à sa place si absent.

**12 sections :**
1. Chargement et profil rapide
2. Prétraitement (log, winsorisation, standardisation)
3. Choix de K : Elbow, Silhouette, Davies-Bouldin
4. K-Means final
5. Comparaison avec Agglomerative et GMM
6. Segmentation RFM par quantiles (référence métier)
7. Visualisation PCA et UMAP
8. Profils des segments
9. Poids par pays (profilage, pas clustering)
10. Stabilité bootstrap
11. Comparaison entre deux snapshots
12. Export et conclusions


In [ ]:
import sys, warnings
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

from src.config import load_config, resolve, load_table
from src.segmentation import (
    SegmentationPreprocessor, evaluate_k_range, suggest_k,
    fit_kmeans, kmeans_metrics, compare_algorithms,
    rfm_segments, name_segments, bootstrap_stability,
    compare_snapshots, build_customer_segments, save_artifacts, load_artifacts,
)

cfg = load_config()
seg_cfg = cfg.get('segmentation', {})
FIG = resolve(cfg, 'figures_dir')
OUT = resolve(cfg, 'outputs_dir')
MODELS = ROOT / seg_cfg.get('models_dir', 'models')
MODELS.mkdir(parents=True, exist_ok=True)

K = seg_cfg.get('k', 5)
K_RANGE = seg_cfg.get('k_range', [2, 3, 4, 5, 6, 7, 8])
SEED = cfg.get('seed', 42)

# ---- Nommage manuel (optionnel) : ecraser les noms automatiques
# NAME_OVERRIDES = {0: 'Champions', 1: 'Fideles', ...}
NAME_OVERRIDES = {}

plt.rcParams.update({'figure.figsize': (10, 4), 'axes.grid': True, 'grid.alpha': 0.3})
print(f'K = {K} | K_RANGE = {K_RANGE} | SEED = {SEED}')

## 1. Chargement et profil rapide

In [ ]:
df = load_table('customer_features', cfg)
print(f'{len(df):,} clients | {df.shape[1]} variables')
print(f"Période : snapshot_date = {df['snapshot_date'].iloc[0]}")
print(f"ipt_mean manquants (clients achat unique) : {df['ipt_mean'].isna().sum():,} ({100*df['ipt_mean'].isna().mean():.1f}%)")
df[['recency', 'frequency', 'monetary', 'gross_revenue', 'avg_basket',
    'n_unique_products', 'return_rate', 'is_wholesale']].describe().round(2)

In [ ]:
# Distribution des 3 variables RFM avant transformation
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, col in zip(axes, ['recency', 'frequency', 'monetary']):
    df[col].clip(upper=df[col].quantile(0.99)).plot(kind='hist', bins=50, ax=ax, title=col)
    ax.set_xlabel(col)
plt.tight_layout()
plt.savefig(FIG / 'rfm_raw_distributions.png', dpi=150)
plt.show()
print('-> Distributions très asymétriques : log-transform nécessaire')

## 2. Prétraitement : log-transform, winsorisation, standardisation

**Règle anti-fuite :** le preprocessor est ajusté UNE SEULE FOIS sur `customer_features` (tout l'historique, T = fin des données). Pour la comparaison de snapshots (section 11), un preprocessor dédié sera ajusté sur le snapshot T1 uniquement.

In [ ]:
# Retirer les colonnes de pays AVANT le clustering (elles serviront au profilage, section 9)
COUNTRY_COLS = ['country_group', 'is_uk']
df_country = df[['CustomerID'] + COUNTRY_COLS].copy()
df_clust = df.drop(columns=COUNTRY_COLS)

log_vars = seg_cfg.get('log_vars', None)
winsor_q = seg_cfg.get('winsor_quantile', 0.99)

preprocessor = SegmentationPreprocessor(log_vars=log_vars, winsor_quantile=winsor_q)
X = preprocessor.fit_transform(df_clust)
print(f'Matrice prete pour le clustering : {X.shape}')
print(f'Variables utilisees ({preprocessor.n_features}) : {preprocessor.feature_names_}')

In [ ]:
# Verifier : zero-mean, unit-std
print(f'Moyenne des colonnes (max abs) : {np.abs(X.mean(axis=0)).max():.2e}')
print(f'Ecart-type des colonnes (min / max) : {X.std(axis=0).min():.4f} / {X.std(axis=0).max():.4f}')

# Distribution apres transformation
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
feature_names = preprocessor.feature_names_
for ax, col in zip(axes, ['recency', 'frequency', 'monetary']):
    if col in feature_names:
        idx = feature_names.index(col)
        pd.Series(X[:, idx]).plot(kind='hist', bins=50, ax=ax, title=f'{col} (log + winsor + std)')
plt.tight_layout()
plt.savefig(FIG / 'rfm_preprocessed.png', dpi=150)
plt.show()

## 3. Choix de K : Elbow, Silhouette, Davies-Bouldin

> **Note importante :** les données RFM forment un continuum ; il n'existe pas de K 'naturel'. L'objectif Silhouette > 0,30 (fixé en phase 1) peut ne pas être atteint à K > 2. Le choix final de K doit combiner les métriques ET l'exploitabilité métier des segments. **Documenter le choix dans la section 'À retenir'.**

In [ ]:
print(f'Evaluation de K dans {K_RANGE} (peut prendre ~30s)...')
metrics_df = evaluate_k_range(X, K_RANGE, random_state=SEED)
k_auto = suggest_k(metrics_df)
print(metrics_df.to_string(index=False))
print(f'\n-> K suggere automatiquement (meilleur Silhouette) : {k_auto}')
print(f'-> K retenu dans config.yaml : {K}')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
metrics_df.plot(x='k', y='inertia', marker='o', ax=axes[0], title='Elbow (inertie)', legend=False)
axes[0].set_xlabel('K'); axes[0].set_ylabel('Inertie')
metrics_df.plot(x='k', y='silhouette', marker='o', ax=axes[1], title='Silhouette (↑ meilleur)', legend=False, color='green')
axes[1].axhline(0.30, color='red', ls='--', label='objectif 0.30')
axes[1].axvline(K, color='orange', ls='--', label=f'K retenu = {K}')
axes[1].legend(); axes[1].set_xlabel('K')
metrics_df.plot(x='k', y='davies_bouldin', marker='o', ax=axes[2], title='Davies-Bouldin (↓ meilleur)', legend=False, color='red')
axes[2].set_xlabel('K')
plt.tight_layout()
plt.savefig(FIG / 'choix_k.png', dpi=150)
plt.show()

## 4. K-Means final (K = {K})


In [ ]:
km = fit_kmeans(X, k=K, random_state=SEED)
labels = km.predict(X)
metrics = kmeans_metrics(X, labels)
print(f'K = {K} | Silhouette = {metrics["silhouette"]} | Davies-Bouldin = {metrics["davies_bouldin"]}')
if metrics['silhouette'] < 0.30:
    print('-> Silhouette < 0.30 : les données RFM forment un continuum.'
          ' K est un choix métier, pas une frontière naturelle. A citer dans les limites.')

# Repartition
df_seg = df_clust.copy()
df_seg['segment_id'] = labels
counts = df_seg.groupby('segment_id').size().rename('n_clients')
pct = (100 * counts / counts.sum()).round(1)
print('\nRepartition des clients par segment :')
print(pd.concat([counts, pct.rename('%')], axis=1))

In [ ]:
# Nommage des segments
SEGMENT_NAMES = name_segments(df_seg, name_overrides=NAME_OVERRIDES)
df_seg['segment_name'] = df_seg['segment_id'].map(SEGMENT_NAMES)
print('Noms attribues aux segments :')
for sid, sname in sorted(SEGMENT_NAMES.items()):
    n = (df_seg['segment_id'] == sid).sum()
    pct_s = 100 * n / len(df_seg)
    print(f'  {sid} -> {sname:30s} ({n:,} clients, {pct_s:.1f}%)')
print('\n-> Pour changer un nom : modifier NAME_OVERRIDES en debut de notebook.')

## 5. Comparaison avec Agglomerative Clustering et GMM

In [ ]:
print('Comparaison des algorithmes (peut prendre ~20s)...')
comp = compare_algorithms(X, k=K, random_state=SEED)
print(comp.to_string(index=False))
print('\n-> ARI vs KMeans mesure la similarite des partitions (1 = identique, 0 = aleatoire).')
print('   Un ARI faible avec Ward ou GMM signifie que les frontieres dependent de l algorithme')
print('   (attendu sur un continuum RFM). A citer dans les limites.')

## 6. Segmentation RFM par quantiles (référence métier)

La segmentation RFM par quantiles est une **référence métier** : elle n'utilise pas de ML, est totalement interprétable et sert de baseline pour comparer avec K-Means.

In [ ]:
rfm = rfm_segments(df, n_quantiles=4)
print('Segments RFM par quantiles :')
print(rfm['rfm_segment'].value_counts())
print('\nComparaison K-Means vs RFM (ARI) :')
from sklearn.metrics import adjusted_rand_score
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
rfm_enc = le.fit_transform(rfm['rfm_segment'])
ari_rfm = adjusted_rand_score(labels, rfm_enc)
print(f'ARI K-Means vs RFM : {ari_rfm:.3f}')
print('-> ARI < 0.5 signifie que K-Means capture une information differente du RFM pur.')
print('   Les variables comportementales (avg_basket, n_unique_products) jouent un role.')

In [ ]:
# Profil RFM
rfm_profile = df.join(rfm[['rfm_segment']]).groupby('rfm_segment')[['recency', 'frequency', 'monetary']].median().round(1)
print(rfm_profile)

## 7. Visualisation PCA et UMAP

In [ ]:
from sklearn.decomposition import PCA
pca = PCA(n_components=2, random_state=SEED)
coords_pca = pca.fit_transform(X)
print(f'Variance expliquee par les 2 axes PCA : {100*pca.explained_variance_ratio_.sum():.1f}%')

colors = plt.cm.tab10.colors
fig, ax = plt.subplots(figsize=(10, 7))
for sid, sname in sorted(SEGMENT_NAMES.items()):
    mask = labels == sid
    ax.scatter(coords_pca[mask, 0], coords_pca[mask, 1],
               label=f'{sname} ({mask.sum():,})', alpha=0.5, s=8,
               color=colors[sid % len(colors)])
ax.legend(markerscale=3, fontsize=9)
ax.set_title('Segments K-Means — espace PCA 2D')
ax.set_xlabel(f'PC1 ({100*pca.explained_variance_ratio_[0]:.1f}%)')
ax.set_ylabel(f'PC2 ({100*pca.explained_variance_ratio_[1]:.1f}%)')
plt.tight_layout()
plt.savefig(FIG / 'segments_pca.png', dpi=150)
plt.show()

In [ ]:
# UMAP (si disponible) ou t-SNE
try:
    import umap
    reducer = umap.UMAP(n_components=2, random_state=SEED, n_neighbors=30, min_dist=0.1)
    coords_low = reducer.fit_transform(X)
    viz_title = 'Segments K-Means — espace UMAP 2D'
    fig_name = 'segments_umap.png'
    print('UMAP utilise.')
except ImportError:
    from sklearn.manifold import TSNE
    sample = min(3000, len(X))
    rng = np.random.default_rng(SEED)
    idx = rng.choice(len(X), size=sample, replace=False)
    coords_low = np.zeros((len(X), 2))
    coords_low[idx] = TSNE(n_components=2, random_state=SEED, perplexity=30).fit_transform(X[idx])
    viz_title = 'Segments K-Means — espace t-SNE 2D (3000 clients)'
    fig_name = 'segments_tsne.png'
    print('UMAP absent : t-SNE utilise sur 3000 clients.')

fig, ax = plt.subplots(figsize=(10, 7))
for sid, sname in sorted(SEGMENT_NAMES.items()):
    mask = labels == sid
    ax.scatter(coords_low[mask, 0], coords_low[mask, 1],
               label=f'{sname} ({mask.sum():,})', alpha=0.5, s=8,
               color=colors[sid % len(colors)])
ax.legend(markerscale=3, fontsize=9)
ax.set_title(viz_title)
plt.tight_layout()
plt.savefig(FIG / fig_name, dpi=150)
plt.show()

## 8. Profils des segments

Les variables suivantes sont calculées **par segment** pour interpréter les groupes. Les noms sont ajustés en conséquence.

In [ ]:
PROFILE_VARS = ['recency', 'frequency', 'monetary', 'avg_basket',
                'n_unique_products', 'return_rate', 'is_wholesale',
                'tenure_days', 'trend', 'ipt_mean']

profile = df_seg.groupby('segment_name')[PROFILE_VARS].agg(['median', 'mean']).round(2)
print(profile.to_string())

In [ ]:
# Tableau de bord segments : taille + CA
summary = df_seg.groupby('segment_name').agg(
    n_clients=('CustomerID', 'count'),
    pct_clients=('CustomerID', lambda x: round(100 * len(x) / len(df_seg), 1)),
    ca_total=('monetary', 'sum'),
    pct_ca=('monetary', lambda x: round(100 * x.sum() / df_seg['monetary'].sum(), 1)),
    recency_median=('recency', 'median'),
    frequency_median=('frequency', 'median'),
).sort_values('ca_total', ascending=False)
print(summary.to_string())

In [ ]:
# Graphiques de profils
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.flatten()
vars_to_plot = ['recency', 'frequency', 'monetary', 'avg_basket', 'n_unique_products', 'return_rate']
for i, var in enumerate(vars_to_plot):
    ax = axes[i]
    vals = [df_seg.loc[df_seg['segment_id'] == sid, var].clip(upper=df_seg[var].quantile(0.95))
            for sid in sorted(SEGMENT_NAMES.keys())]
    ax.boxplot(vals, labels=[SEGMENT_NAMES[sid][:12] for sid in sorted(SEGMENT_NAMES.keys())],
               patch_artist=True,
               boxprops=dict(facecolor='lightblue'))
    ax.set_title(var)
    ax.tick_params(axis='x', rotation=15)
plt.suptitle('Profils des segments (box-plots, valeurs clip au 95e pct)', y=1.01)
plt.tight_layout()
plt.savefig(FIG / 'profils_segments.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Poids par pays (profilage, PAS clustering)

Le pays **n'a pas servi** dans le clustering (interdit, voir `src/segmentation.py`). Il est utilisé ici uniquement pour **profiler** les segments après coup.

In [ ]:
df_seg_country = df_seg.merge(df_country, on='CustomerID', how='left')
country_profile = df_seg_country.groupby(['segment_name', 'country_group']).size().unstack(fill_value=0)
country_pct = country_profile.div(country_profile.sum(axis=1), axis=0).round(3) * 100
print('Répartition géographique par segment (%) :')
print(country_pct.round(1).to_string())

country_pct.plot(kind='bar', stacked=True, figsize=(12, 5), colormap='Pastel1')
plt.title('Répartition géographique par segment (%)')
plt.xticks(rotation=15, ha='right')
plt.ylabel('%')
plt.tight_layout()
plt.savefig(FIG / 'segments_pays.png', dpi=150)
plt.show()

## 10. Stabilité bootstrap

**Protocole :** 50 tirages de 80 % des clients. Sur chaque tirage, un nouveau K-Means est ajusté. L'ARI entre ce modèle et la référence mesure si les frontières sont stables. L'objectif est ARI moyen > 0,80.

In [ ]:
n_boot = seg_cfg.get('n_bootstrap', 50)
boot_frac = seg_cfg.get('bootstrap_sample_frac', 0.80)
print(f'Bootstrap : {n_boot} tirages, {int(boot_frac*100)}% des clients chacun...')
stability = bootstrap_stability(df_clust, preprocessor, k=K,
                                n_bootstrap=n_boot, sample_frac=boot_frac,
                                random_state=SEED)
print(f'ARI moyen = {stability["ari_mean"]} (ecart-type = {stability["ari_std"]})')
print(f'Objectif ARI > 0.80 : {"ATTEINT" if stability["ari_mean"] >= 0.80 else "NON ATTEINT"}')
print('\nJaccard par segment :')
for seg, jac in stability['jaccard_per_segment'].items():
    sname = SEGMENT_NAMES.get(seg, f'Segment {seg}')
    print(f'  {sname:30s} : {jac:.3f} {"OK" if jac >= 0.85 else "< 0.85"}')

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(stability['ari_values'], bins=20, edgecolor='white')
ax.axvline(stability['ari_mean'], color='red', ls='--', label=f'moyenne = {stability["ari_mean"]}')
ax.axvline(0.80, color='green', ls=':', label='objectif = 0.80')
ax.set_title(f'Distribution des ARI bootstrap (n = {n_boot})')
ax.set_xlabel('ARI')
ax.legend()
plt.tight_layout()
plt.savefig(FIG / 'bootstrap_ari.png', dpi=150)
plt.show()

## 11. Comparaison entre deux snapshots

On compare la partition apprise au snapshot `2010-12-31` (train) appliquée au snapshot `2011-09-09` (test) versus un modèle réajusté sur `2011-09-09`.

In [ ]:
# Charger le churn_dataset pour recuperer les features par snapshot
churn_ds = load_table('churn_dataset', cfg)

# T1 = dernier snapshot train, T2 = dernier snapshot test
train_dates = churn_ds[churn_ds['split'] == 'train']['snapshot_date'].unique()
test_dates = churn_ds[churn_ds['split'] == 'test']['snapshot_date'].unique()
print('Snapshots train :', sorted(train_dates))
print('Snapshots test  :', sorted(test_dates))

In [ ]:
from src.features import build_features
from src.config import load_config

transactions = load_table('transactions_clean', cfg)

T1 = pd.Timestamp('2010-12-31')
T2 = pd.Timestamp('2011-09-09')

print(f'Construction des features au snapshot T1 = {T1.date()}...')
df_t1 = build_features(transactions, T1, cfg)
df_t1_clust = df_t1.drop(columns=[c for c in ['country_group', 'is_uk'] if c in df_t1.columns], errors='ignore')

print(f'Construction des features au snapshot T2 = {T2.date()}...')
df_t2 = build_features(transactions, T2, cfg)
df_t2_clust = df_t2.drop(columns=[c for c in ['country_group', 'is_uk'] if c in df_t2.columns], errors='ignore')

print(f'Clients T1 : {len(df_t1):,} | Clients T2 : {len(df_t2):,}')

In [ ]:
# Preprocessor ajuste sur T1 uniquement
prep_t1 = SegmentationPreprocessor(log_vars=log_vars, winsor_quantile=winsor_q)
prep_t1.fit(df_t1_clust)

snap_res = compare_snapshots(df_t1_clust, df_t2_clust, prep_t1, k=K, random_state=SEED)
print(f'Comparaison snapshots T1={T1.date()} vs T2={T2.date()}')
print(f'ARI : {snap_res["ari"]}')
print(f'% clients dans le meme segment : {100*snap_res["pct_same_segment"]:.1f}%')
print(f'Clients communs : {snap_res["n_common_customers"]:,}')

## 12. Export et conclusions

**Livrables :**
- `data/processed/customer_segments.csv` : table client avec segment_id, segment_name, coordonnées PCA
- `models/kmeans.joblib` : modèle K-Means ajusté
- `models/scaler.joblib` : preprocessor ajusté
- `models/segment_meta.json` : métadonnées (K, noms, métriques)

In [ ]:
# Table customer_segments
seg_table = build_customer_segments(df_clust, preprocessor, km, name_overrides=NAME_OVERRIDES)
out_path = Path(ROOT) / 'data/processed/customer_segments.csv'
seg_table.to_csv(out_path, index=False)
print(f'customer_segments.csv sauvegarde : {out_path} ({len(seg_table):,} lignes)')
seg_table.head()

In [ ]:
# Sauvegarde des modeles
artifact_paths = save_artifacts(preprocessor, km, SEGMENT_NAMES, metrics, cfg)
for name, path in artifact_paths.items():
    print(f'{name:10s} -> {path}')

In [ ]:
print('=' * 60)
print('RESULTATS DE LA SEGMENTATION')
print('=' * 60)
print(f'K retenu : {K}')
print(f'Silhouette : {metrics["silhouette"]} | Davies-Bouldin : {metrics["davies_bouldin"]}')
print(f'ARI bootstrap moyen : {stability["ari_mean"]} (objectif > 0.80 : {"atteint" if stability["ari_mean"] >= 0.80 else "non atteint"})')
print()
print(summary[['n_clients', 'pct_clients', 'pct_ca', 'recency_median', 'frequency_median']].to_string())
print()
print('A RETENIR (a completer) :')
print('- Silhouette a K=... : ...')
print('- K suggere auto vs K retenu : ...')
print('- Segment le plus rentable : ...')
print('- Stabilite ARI moyen : ...')
print('- Comparaison snapshots : ARI = ..., % meme segment = ...')
print('- Limites : continuum RFM, methodes divergent (ARI Ward = ..., GMM = ...)')